# AE3 Transformación y enriquecimiento de alquileres de bicicletas

**Continuidad con AE2.** Un equipo hipotético de planeación operativa necesita conocer cómo se distribuyen los alquileres registrados por hora entre días laborables y no laborables en Capital Bikeshare durante 2011 y 2012. La información permite identificar horarios que merecen revisión; no demuestra escasez de bicicletas ni problemas operativos comprobados.

Se mantiene la pregunta central de AE2, el periodo, la unidad de análisis (una hora agregada del sistema), el dataset y su revisión. Se desglosa la pregunta en tres preguntas descriptivas para organizar las tablas de AE4:

| Pregunta | Variables necesarias | Tabla y resultado esperado |
|---|---|---|
| P1 ¿Cómo varían el promedio y la mediana de alquileres por hora entre tipos de día? | `hr`, `workingday`, `cnt`; `dteday` para cobertura | 48 combinaciones hora × tipo de día, con medidas y observaciones |
| P2 ¿Qué franjas de seis horas concentran más alquileres dentro de cada tipo de día? | Hora, tipo de día, alquileres; franja derivada | 8 combinaciones franja × tipo de día, con total y participación |
| P3 ¿Se mantienen los horarios de mayor promedio al separar 2011 y 2012? | Fecha, hora, tipo de día, alquileres; año derivado | 96 combinaciones año × hora × tipo, con máximos y cobertura |

**Justificación del desglose.** P1 desarrolla la comparación prevista en AE2. P2 agrupa el mismo perfil en intervalos iguales para facilitar su lectura operativa; las franjas son una regla de este proyecto. P3 comprueba si el perfil conjunto oculta diferencias entre los dos años. No se añaden fuentes ni se amplía la población. `casual` y `registered` verifican la composición del total; no son usuarios únicos.

Este avance produce datos y tablas; la presentación gráfica final corresponde a AE4.

## 1 Preparación del entorno y fuente

Ejecutar en orden con Python 3.12 y las dependencias de `requirements.txt`, desde la raíz del repositorio o `AE3/notebooks`. Las rutas de salida son relativas al proyecto. El módulo auxiliar `AE3/src/analisis_ae3.py` contiene las funciones de transformación, agregación y verificación utilizadas aquí.

In [1]:
from pathlib import Path
import sys
import json
import hashlib

import pandas as pd
from datasets import load_dataset
from IPython.display import display, Markdown
from pandas.testing import assert_frame_equal

RAIZ = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "requirements.txt").is_file() and (p / "AE3").is_dir()), None)
if RAIZ is None:
    raise FileNotFoundError("Inicia el notebook dentro del repositorio.")
sys.path.insert(0, str(RAIZ / "AE3" / "src"))
from analisis_ae3 import (
    DATASET_ID, REVISION, CONFIGURACION, SPLIT, PREGUNTAS,
    calidad_original, preparar, responder, interpretar,
    exportar_y_verificar, huella_dataframe,
)

print("Dataset:", DATASET_ID)
print("Configuración:", CONFIGURACION, "| Split:", SPLIT)
print("Revisión:", REVISION)
display(pd.DataFrame(PREGUNTAS))

Dataset: t22000t/bike-sharing-tabular
Configuración: default | Split: train
Revisión: 6b645e0cedbc5208fcca11ac94544b088478bf61


,id,pregunta,variables,detalle,resultado
0,P1,¿Cómo varían el promedio y la mediana de alqui...,"hora, tipo_dia, alquileres_total",hora × tipo de día (48 filas),"Perfil horario con media, mediana, cantidad de..."
1,P2,¿Qué franjas de seis horas concentran más alqu...,"franja_horaria, orden_franja, tipo_dia, alquil...",franja de seis horas × tipo de día (8 filas),"Totales, participación dentro del tipo de día ..."
2,P3,¿Se mantienen los horarios de mayor promedio a...,"anio, hora, tipo_dia, alquileres_total",año × hora × tipo de día (96 filas),"Comparación anual de perfiles horarios, máximo..."


In [2]:
dataset = load_dataset(DATASET_ID, name=CONFIGURACION, split=SPLIT,
                       revision=REVISION, token=False)
# Copia de referencia que nunca se sobrescribe ni modifica.
df_original = dataset.to_pandas().copy(deep=True)
huella_original = huella_dataframe(df_original)

ficha_ae2 = json.loads((RAIZ / "AE2" / "resultados" / "ficha_tecnica.json").read_text(encoding="utf-8"))
assert huella_original == ficha_ae2["sha256_dataframe_y_esquema"]
assert df_original.shape == (17379, 17)
print("Original:", df_original.shape, "| misma huella que el DataFrame de AE2:", True)
display(df_original.head())

Original: (17379, 17) | misma huella que el DataFrame de AE2: True


,instant,dteday,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt
0,1,2011-01-01,1,0,1,0,0,6,0,1,0.24,0.2879,0.81,0.0,3,13,16
1,2,2011-01-01,1,0,1,1,0,6,0,1,0.22,0.2727,0.80,0.0,8,32,40
2,3,2011-01-01,1,0,1,2,0,6,0,1,0.22,0.2727,0.80,0.0,5,27,32
3,4,2011-01-01,1,0,1,3,0,6,0,1,0.24,0.2879,0.75,0.0,3,10,13
4,5,2011-01-01,1,0,1,4,0,6,0,1,0.24,0.2879,0.75,0.0,0,1,1


## 2 Reglas de calidad y decisiones

Se revisan nulos, duplicados, rangos y coherencia de fecha, calendario y conteos. Si aparece una incidencia, el flujo se detiene para revisar su tratamiento. **Cero incidencias significa que ese control no encontró problemas**, no que los alquileres valgan cero.

En la revisión de AE2 se identificaron horas sin fila y ceros meteorológicos no explicados por la fuente. No se imputan ni se eliminan registros. Las variables meteorológicas no son necesarias para estas preguntas y no se seleccionan en el DataFrame final; permanecen intactas en `df_original`. Las comprobaciones estructurales no garantizan la exactitud de cada medición original.

In [3]:
controles_original = calidad_original(df_original)
display(controles_original)
assert controles_original["incidencias"].sum() == 0
print("Ceros de humedad en el original:", int(df_original["hum"].eq(0).sum()))
print("No se sustituyen por nulos, promedios ni otros valores.")

,control,incidencias,tratamiento
0,Celdas nulas en las 17 columnas,0,Conservar; no se detectan incidencias
1,Fechas no interpretables,0,Conservar; no se detectan incidencias
2,Filas completas duplicadas,0,Conservar; no se detectan incidencias
3,Duplicados fecha y hora,0,Conservar; no se detectan incidencias
4,Horas fuera de 0 a 23,0,Conservar; no se detectan incidencias
5,Códigos workingday fuera de 0 y 1,0,Conservar; no se detectan incidencias
6,Conteos negativos,0,Conservar; no se detectan incidencias
7,Discrepancias cnt y componentes,0,Conservar; no se detectan incidencias
8,Año codificado distinto de la fecha,0,Conservar; no se detectan incidencias
9,Mes codificado distinto de la fecha,0,Conservar; no se detectan incidencias


Ceros de humedad en el original: 22
No se sustituyen por nulos, promedios ni otros valores.


## 3 Transformación y comparación antes y después

Se seleccionan seis columnas fuente: `dteday`, `hr`, `workingday`, `cnt`, `casual`, `registered`. Se renombran para expresar su significado en español, se interpreta la fecha con formato explícito y el indicador 0/1 se convierte en booleano. Los conteos conservan sus unidades originales de alquileres por registro horario.

Se excluyen once columnas del DataFrame de trabajo por no aportar a las preguntas o por ser redundantes. `yr` y `mnth` se contrastan antes contra la fecha; el año de trabajo se deriva de ella. **No se filtran filas:** se conservan 17.379 registros. La implementación está en `preparar`, cuyo código acompaña al notebook.

In [4]:
df_preparado, calendario, auxiliares = preparar(df_original)
display(auxiliares["comparacion_antes_despues"])
display(auxiliares["tipos_y_nulos_antes_despues"])
display(auxiliares["bitacora_transformaciones"])
assert_frame_equal(df_original, dataset.to_pandas())
assert huella_dataframe(df_original) == huella_original
print("Original sin modificar; filas y total de alquileres conservados.")

,version,filas,columnas,celdas_nulas,total_alquileres
0,Original AE2,17379,17,0,3292679
1,Preparado AE3,17379,11,0,3292679


,version,variable,tipo,nulos
0,Original AE2,instant,int64,0
1,Original AE2,dteday,str,0
2,Original AE2,season,int64,0
3,Original AE2,yr,int64,0
4,Original AE2,mnth,int64,0
5,Original AE2,hr,int64,0
6,Original AE2,holiday,int64,0
7,Original AE2,weekday,int64,0
8,Original AE2,workingday,int64,0
9,Original AE2,weathersit,int64,0


,operacion,registros_afectados_o_incidencias,regla_y_justificacion
0,Selección de seis variables fuente,17379,Se excluyen 11 columnas del DataFrame de traba...
1,Nombres descriptivos en español,17379,Se renombran seis columnas sin alterar sus val...
2,Fecha de texto a datetime64,17379,dteday con formato %Y-%m-%d; errores=raise. No...
3,Indicador laborable de entero a bool,17379,0 a False y 1 a True tras comprobar el dominio.
4,Cinco variables derivadas,17379,"fecha_hora, anio, tipo_dia, orden_franja y fra..."
5,"Nulos, duplicados e inconsistencias",0,"No se eliminan registros, no se imputan ni ree..."
6,Horas sin fila en calendario,165,"Se contabilizan como ausencias, no se agregan ..."
7,Humedad igual a cero en el original,22,Se conserva el original. La humedad no se sele...


Original sin modificar; filas y total de alquileres conservados.


## 4 Enriquecimiento y diccionario actualizado

Las cinco variables incorporadas se calculan exclusivamente con datos existentes:

- `fecha_hora = fecha + hora`: clave para identificar la observación y contrastar cobertura.
- `anio = año(fecha)`: permite separar 2011 y 2012 en P3.
- `tipo_dia`: etiqueta `Laborable` si `es_laborable` es verdadero; `No laborable` en otro caso.
- `orden_franja = hora // 6`: orden de los cuatro intervalos de P2 y del futuro eje gráfico.
- `franja_horaria`: etiquetas `00 a 05`, `06 a 11`, `12 a 17`, `18 a 23`. Todos los intervalos tienen seis horas nominales.

Las etiquetas no son grupos descubiertos ni una clasificación oficial del operador. El CSV no conserva los tipos categóricos y de fecha de Pandas; sus etiquetas y valores se serializan y el diccionario conserva el esquema para volver a interpretarlos.

In [5]:
display(auxiliares["diccionario_variables"])
display(df_preparado.head())
print("Tipos finales:")
print(df_preparado.dtypes.to_string())

,variable,significado_o_regla,tipo_pandas,unidad_o_codigos,utilidad
0,fecha,Fecha del registro; dteday interpretado con fo...,datetime64[us],Fecha,P1 P2 P3 y cobertura
1,hora,"Hora original hr, entero entre 0 y 23.",int64,Hora del día,P1 P3
2,es_laborable,workingday convertido a booleano; True si no e...,bool,True o False,P1 P2 P3
3,alquileres_total,Conteo original cnt en una hora del sistema.,int64,Alquileres por registro horario,P1 P2 P3
4,alquileres_ocasionales,Conteo original casual; no representa personas...,int64,Alquileres por registro horario,Comprobar el total
5,alquileres_registrados,Conteo original registered; no representa pers...,int64,Alquileres por registro horario,Comprobar el total
6,fecha_hora,fecha + hora expresada en horas; sin asignar z...,datetime64[us],Fecha y hora sin zona,Clave única y cobertura
7,anio,Año calendario extraído de fecha; se contrasta...,int64,Año,P3
8,tipo_dia,Etiqueta de es_laborable: Laborable o No labor...,category,Categoría,P1 P2 P3 y leyendas AE4
9,orden_franja,División entera hora // 6; códigos de 0 a 3.,int64,Código ordinal,Orden de P2 y eje AE4


,fecha,hora,es_laborable,alquileres_total,alquileres_ocasionales,alquileres_registrados,fecha_hora,anio,tipo_dia,orden_franja,franja_horaria
0,2011-01-01,0,False,16,3,13,2011-01-01 00:00:00,2011,No laborable,0,00 a 05
1,2011-01-01,1,False,40,8,32,2011-01-01 01:00:00,2011,No laborable,0,00 a 05
2,2011-01-01,2,False,32,5,27,2011-01-01 02:00:00,2011,No laborable,0,00 a 05
3,2011-01-01,3,False,13,3,10,2011-01-01 03:00:00,2011,No laborable,0,00 a 05
4,2011-01-01,4,False,1,0,1,2011-01-01 04:00:00,2011,No laborable,0,00 a 05


Tipos finales:
fecha                     datetime64[us]
hora                               int64
es_laborable                        bool
alquileres_total                   int64
alquileres_ocasionales             int64
alquileres_registrados             int64
fecha_hora                datetime64[us]
anio                               int64
tipo_dia                        category
orden_franja                       int64
franja_horaria                  category


## 5 Cobertura y alcance de las comparaciones

Se construye una cuadrícula auxiliar de 24 horas nominales por fecha, sin añadir filas al dataset analizado ni imputar alquileres. La clasificación del día se obtiene por un mapeo de las fechas del propio archivo: se verifica que cada fecha tenga una sola clasificación y que no existan días completos sin clasificación. No se incorpora otra fuente ni se hace una unión que multiplique registros.

La cuadrícula es un control nominal; no asigna zona horaria ni ajusta cambios de horario de verano no documentados. `cobertura_pct = horas observadas / horas nominales esperadas × 100`. Cada tabla muestra su denominador. La media se calcula sobre las horas **registradas**, no sobre la cuadrícula completa.

In [6]:
display(auxiliares["cobertura_por_anio_tipo"].round(3))
print("Filas de la cuadrícula auxiliar:", len(calendario))
print("Horas sin registro:", int((~calendario["tiene_registro"]).sum()))
assert len(df_preparado) == len(df_original)
print("Filas del dataset final:", len(df_preparado), "(la cuadrícula no se incorpora)")

,anio,tipo_dia,horas_esperadas,horas_observadas,horas_sin_registro,cobertura_pct
0,2011,Laborable,6000,5911,89,98.517
1,2011,No laborable,2760,2734,26,99.058
2,2012,Laborable,6000,5954,46,99.233
3,2012,No laborable,2784,2780,4,99.856


Filas de la cuadrícula auxiliar: 17544
Horas sin registro: 165
Filas del dataset final: 17379 (la cuadrícula no se incorpora)


## 6 P1 Perfil horario por tipo de día

**Pregunta:** ¿Cómo varían el promedio y la mediana de alquileres por hora entre días laborables y no laborables?

**Nivel de detalle:** una fila por hora y tipo de día. `alquileres_total` es la suma de alquileres observados del grupo. El promedio y la mediana se expresan en alquileres por hora registrada; `horas_observadas` es el tamaño del grupo. Se prefiere comparar estas medidas y no solamente totales porque los tipos de día tienen distinto número de registros.

In [7]:
tablas = responder(df_preparado, calendario)
respuestas = interpretar(tablas)
with pd.option_context("display.max_rows", 60, "display.max_columns", None):
    display(tablas["p1_perfil_horario"].round(3))
display(Markdown("**Respuesta P1.** " + respuestas["P1"]))

,tipo_dia,hora,horas_observadas,alquileres_total,promedio_alquileres_por_hora,mediana_alquileres_por_hora,horas_esperadas,horas_sin_registro,cobertura_pct
0,Laborable,0,496,18246,36.786,33.0,500,4,99.200
1,Laborable,1,494,8177,16.553,14.0,500,6,98.800
2,Laborable,2,487,4229,8.684,7.0,500,13,97.400
3,Laborable,3,470,2323,4.943,4.0,500,30,94.000
4,Laborable,4,470,2552,5.430,5.0,500,30,94.000
5,Laborable,5,495,12332,24.913,25.0,500,5,99.000
6,Laborable,6,496,50840,102.500,104.0,500,4,99.200
7,Laborable,7,496,144144,290.613,295.0,500,4,99.200
8,Laborable,8,496,236595,477.006,463.0,500,4,99.200
9,Laborable,9,496,119793,241.518,228.0,500,4,99.200


**Respuesta P1.** Laborable: el mayor promedio corresponde a las 17:00, con 525.29 alquileres por hora registrada, mediana 539.0 y 499 observaciones (cobertura 99.80 %). No laborable: el mayor promedio corresponde a las 13:00, con 372.73 alquileres por hora registrada, mediana 367.0 y 231 observaciones (cobertura 100.00 %). Son franjas para revisar en el escenario operativo, sin inferir faltantes de bicicletas ni causalidad. Las medias describen horas observadas y no se ajustan por clima o año.

## 7 P2 Distribución por franjas de seis horas

**Pregunta:** ¿Qué franjas de seis horas concentran más alquileres dentro de cada tipo de día?

**Nivel de detalle:** una fila por franja y tipo de día. `participacion_alquileres_pct = total de la franja / total del mismo tipo de día × 100`. Cada tipo de día suma 100 % antes del redondeo de presentación. Las cifras se conservan sin redondear para calcular y exportar; solo se redondea la vista.

La participación describe el volumen histórico observado; el promedio por hora y la cobertura aportan contexto frente a las ausencias. No representa demanda insatisfecha ni rentabilidad.

In [8]:
display(tablas["p2_franjas"].round(3))
display(Markdown("**Respuesta P2.** " + respuestas["P2"]))
print("Suma de porcentajes por tipo de día:")
display(tablas["p2_franjas"].groupby("tipo_dia", observed=True)["participacion_alquileres_pct"].sum().to_frame())

,tipo_dia,orden_franja,franja_horaria,horas_observadas,alquileres_total,promedio_alquileres_por_hora,mediana_alquileres_por_hora,horas_esperadas,horas_sin_registro,cobertura_pct,participacion_alquileres_pct
0,Laborable,0,00 a 05,2912,47859,16.435,10.0,3000,88,97.067,2.088
1,Laborable,1,06 a 11,2976,696996,234.206,187.0,3000,24,99.200,30.405
2,Laborable,2,12 a 17,2989,798696,267.212,226.0,3000,11,99.633,34.841
3,Laborable,3,18 a 23,2988,748859,250.622,197.0,3000,12,99.600,32.667
4,No laborable,0,00 a 05,1364,58650,42.999,27.0,1386,22,98.413,5.863
5,No laborable,1,06 a 11,1384,210323,151.967,99.5,1386,2,99.856,21.027
6,No laborable,2,12 a 17,1386,494046,356.455,358.0,1386,0,100.000,49.391
7,No laborable,3,18 a 23,1380,237250,171.920,144.0,1386,6,99.567,23.719


**Respuesta P2.** Laborable: la franja 12 a 17 reúne 798,696 alquileres, 34.84 % del total de ese tipo de día y un promedio de 267.21 por hora registrada. No laborable: la franja 12 a 17 reúne 494,046 alquileres, 49.39 % del total de ese tipo de día y un promedio de 356.45 por hora registrada. La participación usa como denominador los alquileres del mismo tipo de día, no los de todo el dataset. Las franjas tienen seis horas nominales; se informa cobertura para reconocer ausencias.

Suma de porcentajes por tipo de día:


,participacion_alquileres_pct
tipo_dia,
Laborable,100.0
No laborable,100.0


## 8 P3 Comparación del perfil entre 2011 y 2012

**Pregunta:** ¿Se mantienen los horarios de mayor promedio al separar los dos años para cada tipo de día?

**Nivel de detalle:** una fila por año, hora y tipo de día. El máximo se define mediante el mayor promedio horario, no el total. `rango_promedio = 1` identifica las horas de mayor promedio de cada combinación año–tipo de día; si hubiera empate se conservan todas. Se publica la tabla completa de 96 filas y se muestra también el extracto de máximos para facilitar su lectura.

In [9]:
with pd.option_context("display.max_rows", 110, "display.max_columns", None):
    display(tablas["p3_perfil_anual"].round(3))
print("Extracto de máximos; no reemplaza la tabla completa:")
display(tablas["p3_perfil_anual"].loc[tablas["p3_perfil_anual"]["rango_promedio"].eq(1)].round(3))
display(Markdown("**Respuesta P3.** " + respuestas["P3"]))

,anio,tipo_dia,hora,horas_observadas,alquileres_total,promedio_alquileres_por_hora,mediana_alquileres_por_hora,horas_esperadas,horas_sin_registro,cobertura_pct,rango_promedio
0,2011,Laborable,0,247,7243,29.324,27.0,250,3,98.800,19
1,2011,Laborable,1,246,3376,13.724,12.0,250,4,98.400,21
2,2011,Laborable,2,239,1796,7.515,6.0,250,11,95.600,22
3,2011,Laborable,3,229,1000,4.367,4.0,250,21,91.600,24
4,2011,Laborable,4,225,1074,4.773,5.0,250,25,90.000,23
5,2011,Laborable,5,247,4396,17.798,20.0,250,3,98.800,20
6,2011,Laborable,6,248,19218,77.492,87.0,250,2,99.200,17
7,2011,Laborable,7,248,53240,214.677,244.5,250,2,99.200,6
8,2011,Laborable,8,248,86677,349.504,391.5,250,2,99.200,3
9,2011,Laborable,9,248,44800,180.645,189.0,250,2,99.200,8


Extracto de máximos; no reemplaza la tabla completa:


,anio,tipo_dia,hora,horas_observadas,alquileres_total,promedio_alquileres_por_hora,mediana_alquileres_por_hora,horas_esperadas,horas_sin_registro,cobertura_pct,rango_promedio
17,2011,Laborable,17,250,98729,394.916,430.0,250,0,100.0,1
37,2011,No laborable,13,115,32867,285.800,314.0,115,0,100.0,1
65,2012,Laborable,17,249,163391,656.189,704.0,250,1,99.6,1
85,2012,No laborable,13,116,53234,458.914,492.5,116,0,100.0,1


**Respuesta P3.** 2011, Laborable: máximo a las 17:00 (394.92 alquileres por hora registrada). 2011, No laborable: máximo a las 13:00 (285.80 alquileres por hora registrada). 2012, Laborable: máximo a las 17:00 (656.19 alquileres por hora registrada). 2012, No laborable: máximo a las 13:00 (458.91 alquileres por hora registrada). Laborable: el conjunto de horas máximas coincide entre ambos años. No laborable: el conjunto de horas máximas coincide entre ambos años. El máximo se selecciona por la media; la tabla conserva todas las horas y los empates. Esta comparación histórica no demuestra una tendencia futura ni explica sus causas.

## 9 Exportación automática y resumen descriptivo

Se crean las carpetas de salida y se regeneran el dataset preparado, las tres tablas de respuesta, las tablas auxiliares de trazabilidad y el archivo `.txt` **desde los DataFrames de esta ejecución**. Se usa UTF-8, separador coma, punto decimal y no se exporta el índice de Pandas.

El `.txt` incluye dataset/revisión/split, dimensiones originales y finales, columnas/tipos/nulos, estadísticas descriptivas, distribución de categorías, fechas y totales. `validaciones.json` registra respuestas, huellas y archivos generados. La redistribución del dataset preparado está permitida bajo CC BY 4.0 con atribución; las transformaciones de este proyecto quedan documentadas en este notebook y el README.

In [10]:
archivos, validaciones = exportar_y_verificar(
    RAIZ, df_original, df_preparado, tablas, auxiliares, respuestas
)
display(archivos)
print("Verificaciones:")
print(json.dumps(validaciones["verificaciones"], ensure_ascii=False, indent=2))
assert_frame_equal(df_original, dataset.to_pandas())
assert huella_dataframe(df_original) == huella_original

,ruta,filas,columnas,sha256
0,AE3/datos/dataset_preparado.csv,17379.0,11.0,97b9576c55b5e3e8a4aac7fe083bb6a65248654614b881...
1,AE3/resultados/p1_perfil_horario.csv,48.0,9.0,be2d0039dc4446a16cf5eba74c0dee3694d285085a1889...
2,AE3/resultados/p2_franjas.csv,8.0,11.0,ce7c67968bc9106aa07c71369bb8fcd13158739759db63...
3,AE3/resultados/p3_perfil_anual.csv,96.0,11.0,4456958ad22af9fd4aea92eaecef972283e18bf4b0f331...
4,AE3/resultados/controles_calidad.csv,12.0,3.0,ada0ed1972e4c06191280f5da9d833be63999dfa3541c3...
5,AE3/resultados/comparacion_antes_despues.csv,2.0,5.0,5fd280f0525cdf3394810684de57c652e8309a2192be7c...
6,AE3/resultados/tipos_y_nulos_antes_despues.csv,28.0,4.0,198808716264d3ee1f68146762f85c8b6655ebc08fbed8...
7,AE3/resultados/bitacora_transformaciones.csv,8.0,3.0,3230ed841a5e4920f857f62b2e429d5766b7b56fabaf6e...
8,AE3/resultados/diccionario_variables.csv,11.0,5.0,9abdaed7a56f1fc6031f5789c2388cef92e1697cad5485...
9,AE3/resultados/cobertura_por_anio_tipo.csv,4.0,6.0,fd0cec5495c20f84485afd798175af84defc83e61fc916...


Verificaciones:
{
  "original_sin_modificar": true,
  "csv_releidos_comparados_celda_a_celda": true,
  "totales_conservados_en_tres_tablas": true,
  "porcentajes_p2_suman_100_por_tipo": true,
  "resumen_releido_coincide": true
}


In [11]:
ruta_resumen = RAIZ / "AE3" / "resultados" / "resumen_dataframe_transformado.txt"
texto = ruta_resumen.read_text(encoding="utf-8")
assert f"Filas: {len(df_preparado)}" in texto
assert f"Columnas: {len(df_preparado.columns)}" in texto
assert f"Total alquileres: {int(df_preparado['alquileres_total'].sum())}" in texto
for archivo in validaciones["archivos"]:
    assert hashlib.sha256((RAIZ / archivo["ruta"]).read_bytes()).hexdigest() == archivo["sha256"]
print(texto)
print("Exportaciones verificadas: CSV releídos y comparados celda a celda; totales y porcentajes consistentes; TXT coincide con el DataFrame final.")

AE3 RESUMEN DEL DATAFRAME TRANSFORMADO Y ENRIQUECIDO
Dataset: t22000t/bike-sharing-tabular
Revision: 6b645e0cedbc5208fcca11ac94544b088478bf61
Configuracion: default; split: train
Muestra: no. Se conservan todas las filas; se seleccionan variables pertinentes.
Original: 17379 filas y 17 columnas
Filas: 17379
Columnas: 11

COLUMNAS TIPOS Y NULOS
Nota: nulos=0 significa ausencia de celdas vacias, no que todos los valores sean cero.
variable | tipo_pandas | nulos
fecha | datetime64[us] | 0
hora | int64 | 0
es_laborable | bool | 0
alquileres_total | int64 | 0
alquileres_ocasionales | int64 | 0
alquileres_registrados | int64 | 0
fecha_hora | datetime64[us] | 0
anio | int64 | 0
tipo_dia | category | 0
orden_franja | int64 | 0
franja_horaria | category | 0

ESTADISTICAS NUMERICAS
Incluye codigos de hora, anio y orden_franja solo como rangos; no representan cantidades de negocio.
              hora  alquileres_total  alquileres_ocasionales  alquileres_registrados         anio  orden_franja
coun

## 10 Preparación para AE4 y límites

| Evidencia exportada | Uso previsto en AE4 |
|---|---|
| `AE3/datos/dataset_preparado.csv` | Base con fechas, categorías y conteos para filtros y verificación |
| `AE3/resultados/p1_perfil_horario.csv` | Perfil por hora y tipo de día; medias, medianas y cobertura |
| `AE3/resultados/p2_franjas.csv` | Distribución y participación por franja dentro de cada tipo de día |
| `AE3/resultados/p3_perfil_anual.csv` | Comparación por año y tipo de día, con todas las horas |
| `AE3/resultados/diccionario_variables.csv` | Reglas, tipos, unidades y orden de categorías |
| `AE3/resultados/resumen_dataframe_transformado.txt` | Evidencia descriptiva del DataFrame final |

Las tablas identifican horarios observados para una revisión operativa hipotética. No permiten definir cantidades óptimas de bicicletas, analizar estaciones, medir usuarios únicos, solicitudes no atendidas o ingresos, atribuir diferencias al tipo de día ni extrapolar a la operación actual. Las horas ausentes pueden afectar los promedios; mostrar cobertura no corrige ese sesgo. No se analizaron efectos de clima, cambios del servicio ni otros factores.

## Fuentes y condiciones

- [Dataset Hugging Face](https://huggingface.co/datasets/t22000t/bike-sharing-tabular), revisión fijada en el código. Consulta AE3: 2 de noviembre de 2026.
- Fanaee-T, H. (2013). [Bike Sharing, UCI](https://archive.ics.uci.edu/dataset/275/bike+sharing+dataset), DOI [10.24432/C5W894](https://doi.org/10.24432/C5W894). Datos reales agregados de Capital Bikeshare.
- [Licencia CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). Se redistribuyen columnas seleccionadas y variables derivadas, con nombres y tipos transformados; se conservan todas las filas y los conteos.
- [Pandas Group by](https://pandas.pydata.org/docs/user_guide/groupby.html), agrupaciones, medias, conteos y sumas.
- [NIST Measures of Location](https://www.itl.nist.gov/div898/handbook/eda/section3/eda351.htm), interpretación de media y mediana.